In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [3]:
#Current Variables:Business_Name,Entity_Type,Location,Years_in_Business,Industry,Gross_Revenue,Net_Income,Income_Growth_Rate,Taxable_Income,
#                  Operating_Expenses,Depreciation_and_Amortization,Interest_Expenses,R&D_Expenses,Home_Office_Deduction,Number_of_Employees,
#                  Payroll_Expenses,Retirement_Plan_Contributions,Healthcare_Expenses,State_Specific_Tax_Incentives,Sales_Tax_Obligations,
#                  Inventory_Method,Cost_of_Goods_Sold,Fixed_Assets,Capital_Expenditures,Property_Ownership_vs_Leasing,Vehicle_Use,R&D_Credits,
#                  Energy_Efficiency_Credits,Work_Opportunity_Tax_Credit,Employee_Retention_Credits,Outstanding_Debt,Interest_Payments,Loan_Type,
#                  Business_Investments,Retirement_Contributions,Previous_Tax_Liabilities,Tax_Filing_Method,Carryforwards_and_Carrybacks,
#                  Current_Tax_Strategy,Revenue_Streams,Domestic_vs_Foreign_Income,Major_Asset_Sales,Capital_Gains_and_Losses


#Prospective Variables:
#                  Identifiers: Business_ID
#                  Categorical: Industry, Business Structure, Revenue Streams, Property Ownership vs. Leasing, Current Tax Strategy, 
#                  Numeric: Years in Business, Previous Tax Liabilities, Gross Revenue, Net Income, Taxable Income, Income Growth Rate,
#                           Operating Expenses, Payroll Expenses, Retirement Plan Contributions, Healthcare Expenses, Depreciation and Amortization 
#                           Interest Expenses, R&D Expenses, R&D Credits, Energy Efficiency Credits, Work Opportunity Tax Credit (WOTC), 
#                           Employee Retention Credits (ERC) , Carryforwards and Carryback, Home Office Deduction, Fixed Assets, Capital Expenditures, 
#                           Vehicle Use for Business, Major Asset Sales, Capital Gains and Losses, Outstanding Debt, Interest Payments, Loan Type, 
#                           Business Investments, Sales Tax Obligations, Inventory Method, Number of Employees, Employee Salaries & Wages, 
#                           Bonuses and Incentives, Retirement Contributions for Employees, Health Benefits Provided 


In [43]:
#Import Data
dataSet = pd.read_csv('C:\\Users\\bfire\\Desktop\\CSUSpring2025\\Senior_Design\\synthetic_tax_optimization_data_v4.csv')
print(dataSet.info()) 
print(dataSet.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 46 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Business_ID                             500 non-null    int64  
 1   Industry                                500 non-null    object 
 2   Business_Structure                      500 non-null    object 
 3   Revenue_Streams                         500 non-null    object 
 4   Property_Ownership_vs_Leasing           500 non-null    object 
 5   Current_Tax_Strategy                    500 non-null    object 
 6   Years_in_Business                       500 non-null    int64  
 7   Previous_Tax_Liabilities                500 non-null    int64  
 8   Gross_Revenue                           500 non-null    int64  
 9   Net_Income                              500 non-null    int64  
 10  Taxable_Income                          500 non-null    int64 

In [39]:
categorical_cols = [
    "Industry",
    "Business_Structure",
    "Revenue_Streams",
    "Property_Ownership_vs_Leasing",
    "Current_Tax_Strategy",
    "Loan_Type",
    "Inventory_Method"
]

numerical_cols = [
    "Years_in_Business", "Previous_Tax_Liabilities", "Gross_Revenue", "Net_Income",
    "Taxable_Income", "Income_Growth_Rate", "Operating_Expenses", "Payroll_Expenses",
    "Retirement_Plan_Contributions", "Healthcare_Expenses", "Depreciation_and_Amortization",
    "Interest_Expenses", "R&D_Expenses", "R&D_Credits", "Energy_Efficiency_Credits",
    "Work_Opportunity_Tax_Credit (WOTC)", "Employee_Retention_Credits (ERC)",
    "Carryforwards_and_Carryback", "Home_Office_Deduction", "Fixed_Assets",
    "Capital_Expenditures", "Vehicle_Use_for_Business", "Major_Asset_Sales",
    "Capital_Gains_and_Losses", "Outstanding_Debt", "Interest_Payments",
    "Business_Investments", "Sales_Tax_Obligations", "Number_of_Employees",
    "Employee_Salaries_&_Wages", "Bonuses_and_Incentives",
    "Retirement_Contributions_for_Employees", "Health_Benefits_Provided"
]


target_cols = ["R&D_Credit_Eligible", "ERC_Eligible", "WOTC_Eligible", "Energy_Credit_Eligible", "Home_Office_Eligible"]


In [53]:
#Use one-hot encouding for pre-processing Categorical data
df_encoded = pd.get_dummies(dataSet, columns=categorical_cols, drop_first=True)
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])



In [55]:
X = df_encoded.drop(columns=target_cols)  # Features
y = df_encoded[target_cols]  # Labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data preprocessing complete! Ready for model training.")

print(dataSet.info()) 
print(dataSet.head())

Data preprocessing complete! Ready for model training.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 46 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Business_ID                             500 non-null    int64  
 1   Industry                                500 non-null    object 
 2   Business_Structure                      500 non-null    object 
 3   Revenue_Streams                         500 non-null    object 
 4   Property_Ownership_vs_Leasing           500 non-null    object 
 5   Current_Tax_Strategy                    500 non-null    object 
 6   Years_in_Business                       500 non-null    int64  
 7   Previous_Tax_Liabilities                500 non-null    int64  
 8   Gross_Revenue                           500 non-null    int64  
 9   Net_Income                              500 non-null    int64  
 10  Taxable

In [57]:

# Train a separate Random Forest model for each tax credit
models = {}
for target in target_cols:
    print(f"Training model for {target}...")
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train[target])
    
    # Store model
    models[target] = model
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Evaluate
    print(f"Accuracy for {target}: {accuracy_score(y_test[target], y_pred):.4f}")
    print(classification_report(y_test[target], y_pred))
    print("-" * 50)

Training model for R&D_Credit_Eligible...
Accuracy for R&D_Credit_Eligible: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00        89

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

--------------------------------------------------
Training model for ERC_Eligible...
Accuracy for ERC_Eligible: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

--------------------------------------------------
Training model for WOTC_Eligible...
Accuracy for WOTC_Eligible: 0.9800
              precision    recall  f1-score   support

           0       1